In [1]:
# langgraph 패키지 설치
# https://docs.langchain.com/oss/python/langgraph/install

In [2]:
# langchain 패키지 설치

In [ ]:
# langchain-google-genai 패키지 설치
# https://docs.langchain.com/oss/python/integrations/providers/overview


# Google vertex AI vs Google genai

# Google vertex AI는 엔터프라이즈 용도로 사용하는 패키지 
# 예를들어 결제 정보, 할당량 같은 것을 프로덕트 단에서 사용하려고 할 때 사용

# Google genai는 개인적으로 사용하는 패키지
# 예를들어 모델 호출 같은 것을 개인적으로 사용하려고 할 때 사용

# 따라서 개인적으로 사용하는 경우 Google genai를 사용하는 것이 좋다.




In [7]:
# os 환경 변수 GEMINI API KEY 입력
import os
from dotenv import load_dotenv

load_dotenv()

True

# Agent 구축 기초
https://docs.langchain.com/oss/python/langgraph/quickstart

### 1. Model 및 Tool 정의

In [8]:
# 모델 객체 생성
# https://docs.langchain.com/oss/python/integrations/providers/google

from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-preview",
)

ModuleNotFoundError: No module named 'langchain_google_genai'

In [4]:
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a * b


@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b


@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b

In [5]:
# tools 정의

tools = [multiply, add, divide]

In [6]:
# model - tool 바인딩
model_with_tools = model.bind_tools(tools)

### 2. State 정의

In [7]:
# State 정의
# https://docs.langchain.com/oss/python/langgraph/quickstart#2-define-state

from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages

class State(TypedDict):
    # 대화 기록은 리스트 형태로 '누적'되도록 add_messages 리듀서 적용
    messages: Annotated[list[AnyMessage], add_messages]
    # 모델 호출 횟수 기록용
    llm_calls: int


### [add_messages 참조](https://reference.langchain.com/python/langgraph/graphs/?_gl=1*ry73wr*_gcl_au*MTQ0NjcwMDQyNS4xNzYxNzg2NzMz*_ga*MjEyNDg4NjgwNC4xNzYyOTI0NjMz*_ga_47WX3HKKY2*czE3NjgyNzQ0NjEkbzI4JGcxJHQxNzY4Mjc2NzA5JGozMyRsMCRoMA..#langgraph.graph.message.add_messages)

### 3. Node 정의

In [ ]:
# llm_call 노드

from langchain.messages import SystemMessage

def llm_call(state: State):
    """LLM이 현재 상태를 보고 답변하거나, 도구 사용을 요청하는 작업자"""

    # 시스템 메시지를 맨 앞에 추가하고 기존 대화 기록을 뒤에 붙여서 LLM에게 전송
    response = model_with_tools.invoke(
        [
            SystemMessage(
                content="당신은 사칙연산을 완벽하게 해내는 유능한 Agent입니다."
            )
        ] + state["messages"]
    )

    # 작업이 끝나면, 새로 생성된 메시지 1개와 카운터 1 증가분을 공책에 업데이트
    return {
        "messages": [response],
        "llm_calls": state.get('llm_calls', 0) + 1 # 만약에 초기 값이 없으면 0이 되도록 한다 
    }


In [9]:
# tool 목록 생성

from langchain.messages import ToolMessage

# 도구 이름을 키로, 실제 함수를 값으로 가지는 딕셔너리 준비
tools_by_name = {tool.name: tool for tool in tools}

def tool_node(state: State):
    """LLM이 도구 사용을 요청했을 때, 실제로 함수를 실행하는 작업자"""

    result = []
    # 공책의 맨 마지막 메시지(LLM의 요청장)에서 tool_calls 리스트 꺼내기
    last_message = state["messages"][-1]

    for tool_call in last_message.tool_calls:
        # 1. 도구 이름으로 실제 파이썬 함수 찾기
        tool = tools_by_name[tool_call["name"]]

        # 2. 인자(args)를 넣고 함수 실행
        tool_result = tool.invoke(tool_call["args"])

        # 3. 실행 결과를 ToolMessage로 포장 (tool_call_id를 반드시 맞춰주어야 모델이 인식함)
        result.append(ToolMessage(content=tool_result, tool_call_id=tool_call["id"]))

    # 도구 실행 결과들을 공책에 업데이트
    return {"messages": result}


In [10]:
# tool_node

tool_calls = [
    {"name": "weather", "args": {"city": "Seoul"}, "id": "call_12345"}, # 서울 요청 (ID: 12345)
    {"name": "weather", "args": {"city": "Tokyo"}, "id": "call_67890"}  # 도쿄 요청 (ID: 67890)
]


In [ ]:
{'messages': [
    HumanMessage(content='42 + 3 * 23은 뭔가요?', additional_kwargs={}, response_metadata={}, id='d668f973-433b-43a6-832c-12f69d7a1d47'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 666, 'prompt_tokens': 271, 'total_tokens': 937, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CYN0BtgC1vSXVOqkym1wLgdCHV0mg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--e3ff5826-b8ae-45ef-b321-845634280cac-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 23}, 'id': 'call_6JIP9XWGEL4Y8gsDf7kLMCns', 'type': 'tool_call'}], usage_metadata={'input_tokens': 271, 'output_tokens': 666, 'total_tokens': 937, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 640}}),
    ToolMessage(content='69', name='multiply', id='ce3260fd-d978-4f22-9ac8-7ecba8497b5c', tool_call_id='call_6JIP9XWGEL4Y8gsDf7kLMCns'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 302, 'total_tokens': 322, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CYN0IzrIXR33PjGJkChQkJNBWEPNp', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--79f83b57-4a2b-4ceb-8dd1-79bb8ac2907c-0', tool_calls=[{'name': 'add', 'args': {'a': 42, 'b': 69}, 'id': 'call_dYzyonAijap4wIkosXukpffe', 'type': 'tool_call'}], usage_metadata={'input_tokens': 302, 'output_tokens': 20, 'total_tokens': 322, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
    ToolMessage(content='111', name='add', id='a30c98b9-41e5-4496-aed3-044c02028346', tool_call_id='call_dYzyonAijap4wIkosXukpffe'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 666, 'prompt_tokens': 333, 'total_tokens': 999, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CYN0KeTPVjeebFxGf8IXHbmsLUKQ9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--3e09231b-dfd0-4190-9c22-024988cc4389-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 23}, 'id': 'call_7i9W8KYCfoyL0V5YM3hA9CNM', 'type': 'tool_call'}], usage_metadata={'input_tokens': 333, 'output_tokens': 666, 'total_tokens': 999, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 640}}),
    ToolMessage(content='69', name='multiply', id='f041ffd3-626e-4541-9ea7-db7c200a49ef', tool_call_id='call_7i9W8KYCfoyL0V5YM3hA9CNM'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 922, 'prompt_tokens': 364, 'total_tokens': 1286, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 896, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CYN0QZAwvbco2nLwS2UlXoKoQJ2zB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--eb606c0c-8c16-4789-928d-2ed8e902074f-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 23}, 'id': 'call_bUU09sNKZb2CMKxPwn3dc7pW', 'type': 'tool_call'}], usage_metadata={'input_tokens': 364, 'output_tokens': 922, 'total_tokens': 1286, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 896}}),
    ToolMessage(content='69', name='multiply', id='9ae49215-1418-4f1b-88e3-c32caa9dd511', tool_call_id='call_bUU09sNKZb2CMKxPwn3dc7pW'),
    AIMessage(content='정답은 111입니다.\n\n설명:\n- 먼저 곱셈의 순서를 따릅니다: 3 * 23 = 69\n- 그리고 남은 더하기를 수행합니다: 42 + 69 = 111\n\n추가로, 계산 과정을 한 번에 확인하는 방법도 보여드리면:\n- 42 + 3 * 23 = 42 + (3 * 23) = 42 + 69 = 111\n\n필요한 다른 연산이 있으면 말씀해 주세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 395, 'total_tokens': 510, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CYN0Ywh5LSATE3IezdrYTcoik9Abu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--79aa9996-99f3-4463-9986-b33c4f53d706-0', usage_metadata={'input_tokens': 395, 'output_tokens': 115, 'total_tokens': 510, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})
    ]
}

### 4. 그래프 생성

In [ ]:
# 그래프 생성

from langgraph.graph import StateGraph, START

# 1. 빈 작업장(그래프) 도면 펼치기 (공책 양식을 알려줌)
agent_builder = StateGraph(State)

# 2. add_node: 도면에 작업자(노드) 배치하기
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# 3. add_edge: 고정된 연결선(Edge) 긋기
# 시작(START)하자마자 무조건 llm_call 작업자에게 최초로 공책을 넘깁니다.
agent_builder.add_edge(START, "llm_call")
# 도구 실행이 끝나면 결과값을 얻었으니 무조건 다시 llm_call 작업자에게 공책을 넘겨 최종 답변 생성
agent_builder.add_edge("tool_node", "llm_call")



In [ ]:
# 조건부 함수 작성

from langgraph.graph import END

def should_continue(state: State):
    """
    LLM의 응답을 보고 다음 단계로 어디를 갈지 결정하는 라우팅 함수
    """
    last_message = state["messages"][-1]

    # LLM이 도구 호출(tool_calls) 요청장을 작성했다면? -> 도구 작업자(tool_node)에게 가라고 문자열 반환
    if last_message.tool_calls:
        return "tool_node"

    # 도구 호출이 없고 최종 답변을 작성했다면? -> 작업 종료(END) 객체 반환
    return END


In [ ]:
# 조건부 엣지 연결

# 4. add_conditional_edges: 라우팅 기능을 하는 선 설치
agent_builder.add_conditional_edges(
    "llm_call",         # 1) 출발지: llm_call 작업이 끝나면 무조건 이 라우터 함수가 발동합니다.
    should_continue,    # 2) 라우팅 함수: 위에서 만든 함수가 공책을 보고 어디로 갈지 판단합니다.
    ["tool_node", END]  # 3) 도착지 목록: 갈 수 있는 목적지들을 리스트나 딕셔너리로 명시합니다.
)


In [ ]:
# 컴파일 (최종 에이전트 생성)
agent = agent_builder.compile()

In [ ]:
agent

### 실행

In [ ]:
from langchain.messages import HumanMessage

messages = [HumanMessage(content="3과 4를 더해줘")]
response = agent.invoke({"messages": messages})

In [ ]:
response

In [ ]:
# 응답 출력

In [ ]:
messages = [HumanMessage(content="3과 4를 더한 뒤 7을 곱해줘.")]
response = agent.invoke({"messages": messages})

In [ ]:
# Gemini 응답 출력


In [ ]:
response["messages"][-1].content[-1]['text']

### OpenAI 기반 Agent

In [ ]:
!pip install -qU langchain-openai

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

In [ ]:
model_with_tools = model.bind_tools(tools)

In [ ]:
from langchain.messages import SystemMessage

def llm_call(state):
    """LLM이 현재 상태를 보고 답변하거나, 도구 사용을 요청하는 Node"""

    # 시스템 메시지를 맨 앞에 추가하고 기존 대화 기록(state["messages"])을 뒤에 붙여서 LLM에게 전송
    response = model_with_tools.invoke(
        [
            SystemMessage(
                content="당신은 사칙연산을 하는 유능한 Agent입니다."
            )
        ] + state["messages"]
    )

    # 변경된 상태를 반환
    return {
        "messages": [response],
        "llm_calls": state.get('llm_calls', 0) + 1

    }

In [ ]:
tools_by_name = {tool.name: tool for tool in tools}

In [ ]:
from langchain.messages import ToolMessage

def tool_node(state):
    """LLM이 도구 사용을 요청했을 때, 실제로 도구를 실행하는 단계"""

    result = []
    # 가장 최근 메시지(LLM의 응답)에서 도구 호출 요청(tool_calls)들을 꺼내기
    for tool_call in state["messages"][-1].tool_calls:
        # 1. 도구 이름으로 실제 함수 찾기
        tool = tools_by_name[tool_call["name"]]

        # 2. 함수를 실행하여 결과를 얻기
        tool_result = tool.invoke(tool_call["args"])

        # 3. 결과를 ToolMessage 형태로 포장 (tool_call_id는 필수!)
        result.append(ToolMessage(content=tool_result, tool_call_id=tool_call["id"]))

    # 실행 결과를 대화 기록에 추가
    return {"messages": result}

In [ ]:
from langgraph.graph import StateGraph, START

# 1. 워크플로우(그래프) 생성
agent_builder = StateGraph(State)

# 2. 노드(작업자) 배치
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# 3. 엣지(연결선) 연결
agent_builder.add_edge(START, "llm_call")
agent_builder.add_edge("tool_node", "llm_call")

In [ ]:
from langgraph.graph import StateGraph, START, END

def should_continue(state: State):
    """
    LLM의 응답을 보고 다음 단계로 어디를 갈지 결정
    """
    messages = state["messages"]
    last_message = messages[-1]

    # LLM이 도구 호출(tool_calls)을 포함한 응답을 보낸 경우
    if last_message.tool_calls:
        return "tool_node"

    # 도구 호출이 없으면 작업 종료
    return END

In [ ]:
# 조건부 엣지 연결
agent_builder.add_conditional_edges(
    "llm_call",         # 출발지
    should_continue,    # 판단 로직 함수
    ["tool_node", END]  # 갈 수 있는 목적지들
)

In [ ]:
# 컴파일 (최종 에이전트 생성)
agent = agent_builder.compile()

In [ ]:
from langchain.messages import HumanMessage

messages = [HumanMessage(content="3과 4를 더해줘")]
response = agent.invoke({"messages": messages})

In [ ]:
response

In [ ]:
# Gemini 기반 Agent 실행 시 response 결과와 다른 것을 확인할 수 있다.

{'messages': [HumanMessage(content='3과 4를 더해줘', additional_kwargs={}, response_metadata={}, id='58bb1e6d-893a-4db4-af20-58a530401047'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'add', 'arguments': '{"b": 4, "a": 3}'}, '__gemini_function_call_thought_signatures__': {'1038d4ae-692e-4145-97da-12bbde2ed3ad': 'EoUBCoIBAXLI2nzToqLq3PDt/3G+Fv0+R9e4f6HBYkqkdWf/71DI1HugthAVPOPq+T1uGvsBJNpAqI9GNDIXSd7tnThiXA8f8IRpKxp2v374N5y0A1YoUbl4JqOlRNTIRTQDqiUt0lXg+I4QKnKhQZ+z8ZuwNQkSPwnZh54H8D+tz9ld63GerQ=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019bbab8-eaa1-7720-948c-a30d5334d5e4-0', tool_calls=[{'name': 'add', 'args': {'b': 4, 'a': 3}, 'id': '1038d4ae-692e-4145-97da-12bbde2ed3ad', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 236, 'output_tokens': 47, 'total_tokens': 283, 'input_token_details': {'cache_read': 0}, 'output_token_d